In [1]:
from tqdm import tqdm
import math
import json
import os
from datetime import datetime

import torch
from torch_geometric.loader import DataLoader
import torch.nn as nn

from extractor import PDBBindOrchestrator
from tokenizer import UniversalPDBBindDataset
from model import UniversalHybridSlotModel
from encoders.original_quantum_encoder import QuantumReUploadingLayer
from encoders.classic_trio_encoder import TrioPipelineFactory
from evaluator import EValuator
from splitter import PDBBindSplitter
from loss_functions.loss_functions import get_loss_function

In [2]:
# 1. ЕДИНЫЙ ГЛОБАЛЬНЫЙ КОНФИГ
config = {
    "experiment_name": "GGG_GATv2_AdamW_SGD",
    "architecture": "GGG", # C-Prot(CNN), C-Lig(CNN), G-Pock(GNN)
    "dataset": {
        "split_strategy": "random",
        "max_prot": 1000,
        "max_lig": 150,
        "max_pock": 63,
        "batch_size": 32,
        "val_frac": 0.15, # 15% от Refined пойдет на валидацию
        "prot_vocab": {c: i+1 for i, c in enumerate("ACDEFGHIKLMNPQRSTVWY")},
        "lig_vocab": {c: i+1 for i, c in enumerate("ABCDEFGHIKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-=[]()#%@+./\\:,;^$")},
    },
    "model": {
        "embed_dim": 128,
        "gnn_mode": "gat_v2",
        "gat_heads": 4,
        "pooling_type": "max", # "max", "mean"
        "cnn_mode": "parallel",
        "n_qubits": 9,
        "q_layers": 8,
        "combine_method": "perturbation",
    },
    "training": {
        "epochs": 20,
        "classic_lr": 0.001,
        "quantum_lr": 0.0001,
        "classic_optimizer": "AdamW",
        "quantum_optimizer": "SGD",
        "loss_fn": "RankingMSELoss",
    }
}

config["dataset"]["prot_vocab"]['?'] = 0
config["dataset"]["lig_vocab"]['?'] = 0

In [3]:
def get_loss_bar(loss_val, bar_len=10):
    # Используем log10 для нормализации динамического диапазона
    # Добавляем 1e-9, чтобы избежать log(0)
    log_loss = math.log10(loss_val + 1e-9)
    
    # Масштабируем: допустим, мы ожидаем лосс от 50 (log ~1.7) до 0.1 (log -1)
    # Сделаем простую линейную закраску для диапазона log_loss от -1 до 2
    level = (log_loss + 1) / 3  # нормализуем в [0, 1]
    level = max(0, min(1, level)) # ограничиваем
    
    filled = int(level * bar_len)
    return "[" + "█" * filled + " " * (bar_len - filled) + "]"

In [4]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_name = f"{config['experiment_name']}_{timestamp}"

exp_run_dir = f"runs/{exp_name}"
exp_data_dir = f"datasets/{exp_name}" # Индивидуальная папка для датасетов!

os.makedirs(exp_run_dir, exist_ok=True)
os.makedirs(exp_data_dir, exist_ok=True)
os.makedirs("data/base_datasets", exist_ok=True) # Глобальный кэш

In [5]:
parsers, trio_encoder = TrioPipelineFactory.build(config["architecture"], config)
prot_parser, lig_parser, pock_parser = parsers

processor = PDBBindOrchestrator(prot_parser=prot_parser, lig_parser=lig_parser, pock_parser=pock_parser)
# processor.extract_subset("refined") # Раскомментировать при первом запуске

df_refined = processor.build_dataset(subset="refined", fmt="pickle", save_dir="data/base_datasets")
# refined_dataset_full_path, refined_metadata_full_path = processor.full_path, processor.full_meta_path
df_core = processor.build_dataset(subset="core", fmt="pickle", save_dir="data/base_datasets")
# core_dataset_full_path, core_metadata_full_path = processor.full_path, processor.full_meta_path
# del df_refined, df_core

Запуск параллельного парсинга на 8 ядрах...


100%|██████████| 4057/4057 [02:46<00:00, 24.36it/s]


Успешно: 4057, Ошибок: 0
Counter()
Метаданные сохранены в data/base_datasets/pdbbind_refined_protG_ligG_pockG_meta.json
Датасет сохранен в data/base_datasets/pdbbind_refined_protG_ligG_pockG.pickle (сжатие: None)
Запуск параллельного парсинга на 8 ядрах...


100%|██████████| 290/290 [00:11<00:00, 25.82it/s]


Успешно: 290, Ошибок: 0
Counter()
Метаданные сохранены в data/base_datasets/pdbbind_core_protG_ligG_pockG_meta.json
Датасет сохранен в data/base_datasets/pdbbind_core_protG_ligG_pockG.pickle (сжатие: None)


In [6]:
# name_train = os.path.basename(refined_dataset_full_path).split(".")[0]
# name_test = os.path.basename(core_dataset_full_path).split(".")[0]

# full_train_path = os.path.join(os.path.dirname(refined_dataset_full_path), f"{name_train}_train.pickle")
# full_test_path = os.path.join(os.path.dirname(core_dataset_full_path), f"{name_test}_test.pickle")

# df_refined = pd.read_pickle(refined_dataset_full_path)
# df_core = pd.read_pickle(core_dataset_full_path)
clean_refined = df_refined[~df_refined['pdb_id'].isin(df_core['pdb_id'])]

train_df, val_df = PDBBindSplitter.split(
    clean_refined,
    strategy=config["dataset"]["split_strategy"], 
    val_frac=config["dataset"]["val_frac"],
    seed=42
)

test_df = df_core

train_path = f"{exp_data_dir}/train.pickle"
val_path   = f"{exp_data_dir}/val.pickle"
test_path  = f"{exp_data_dir}/test_core.pickle"

train_df.to_pickle(train_path)
test_df.to_pickle(test_path)
val_df.to_pickle(val_path)

Split strategy: random


In [7]:
print(f"Эксперимент: {exp_name}")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test (Core): {len(df_core)}")

Эксперимент: GGG_GATv2_AdamW_SGD_20260418_162049
Train: 3202 | Val: 565 | Test (Core): 290


In [8]:
config["dataset"].update({
    "train_path": train_path,
    "val_path": val_path,
    "test_path": test_path,
})

In [9]:
# 1. Инициализируем датасеты, чтобы узнать реальные размеры словарей, длины согласно статье.
ds_kwargs = {
    'max_prot': config['dataset']['max_prot'], 
    'max_lig': config['dataset']['max_lig'], 
    'max_pock': config['dataset']['max_pock']
}

train_ds = UniversalPDBBindDataset(config["dataset"]["train_path"], **ds_kwargs)
test_ds = UniversalPDBBindDataset(config["dataset"]["test_path"], **ds_kwargs)
val_ds   = UniversalPDBBindDataset(config["dataset"]["val_path"], **ds_kwargs)

train_loader = DataLoader(train_ds, batch_size=config['dataset']['batch_size'], shuffle=True)
test_loader = DataLoader(test_ds, batch_size=config['dataset']['batch_size'], shuffle=False)
val_loader   = DataLoader(val_ds, batch_size=config['dataset']['batch_size'], shuffle=False)

config["dataset"]["prot_vocab"] = max(train_ds.prot_vocab.values()) + 1
config["dataset"]["lig_vocab"] = max(train_ds.lig_vocab.values()) + 1

# 3. Инициализация модели и железа
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# HQDeepDTAF = UniversalHybridModel(
#     protein_encoder = FlexCNNBlock(config['dataset']['prot_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
#     ligand_encoder = FlexCNNBlock(config['dataset']['lig_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
#     pocket_encoder = FlexCNNBlock(config['dataset']['prot_vocab'], config['model']['embed_dim'], mode=config['model']['cnn_mode']),
#     quantum_encoder = QuantumReUploadingLayer(config['model']['n_qubits'], config['model']['q_layers'])
# )

CustomNetwork = UniversalHybridSlotModel(
    graph_encoder=trio_encoder,           # Наш CCG мета-энкодер
    classic_pooler=None,                  
    quantum_pooler=None,                  
    global_readout=nn.Identity(),         # Ридаут уже внутри Trio
    quantum_encoder=QuantumReUploadingLayer(config['model']['n_qubits'], config['model']['q_layers']),         
    decider_hidden_dims=[64, 32]
)

model = CustomNetwork.to(device)
evaluator = EValuator(model, device)

# 4. Настройка обучения
# optimizer = optim.Adam(model.parameters(), lr=0.001)
classic_params = (
    list(model.graph_encoder.parameters()) + 
    list(model.classic_decider.parameters()) + 
    list(model.final_mixer.parameters()) +
    list(model.to_quantum_adapter.parameters())
)
quantum_params = (
    list(model.quantum_encoder.parameters()) +
    list(model.quantum_decider.parameters())
)

# CNN и обвязка любят Adam за скорость
classic_opt_class = getattr(torch.optim, config['training']['classic_optimizer'])
classic_optimizer = classic_opt_class(classic_params, lr=config['training']['classic_lr'])
    
# Квантовое ядро часто лучше учится на SGD или Adagrad, 
# так как они меньше "шумят" в фазовом пространстве
# quantum_optimizer = optim.AdamW(quantum_params, lr=0.0001)
quantum_opt_class = getattr(torch.optim, config['training']['quantum_optimizer'])
quantum_optimizer = quantum_opt_class(quantum_params, lr=config['training']['quantum_lr'], momentum=0.9)

criterion = get_loss_function(config['training'])
print(f"Используем Loss: {config['training']['loss_fn']}")

def train(epochs=config['training']['epochs'], show_plots=False, save_plots=True): # По статье авторы используют 20 эпох 
    print(f"Starting training on {device}...")
    # Создаем папку для эксперимента
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    exp_dir = f"runs/{config['experiment_name']}_{timestamp}"
    os.makedirs(exp_dir, exist_ok=True)
    print(f"!!! ФАЙЛЫ СОХРАНЯЮТСЯ СЮДА: {exp_dir} !!!")
    config['dataset']['actual_sizes'] = {
        'train': len(train_ds),
        'val': len(val_ds),
        'test': len(test_ds)
    }
    with open(f"{exp_dir}/config.json", 'w') as f:
        json.dump(config, f, indent=4)

    best_val_r = -1.0
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        # Создаем обертку над лоадером
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", unit="batch", leave=True)
        
        for i, (prot, lig, pock, y) in enumerate(pbar):
            prot = prot.to(device) if hasattr(prot, 'to') else {k: v.to(device) for k, v in prot.items()}
            lig = lig.to(device) if hasattr(lig, 'to') else {k: v.to(device) for k, v in lig.items()}
            pock = pock.to(device) if hasattr(pock, 'to') else {k: v.to(device) for k, v in pock.items()}
            y = y.to(device)
            
            classic_optimizer.zero_grad()
            quantum_optimizer.zero_grad()
            output = model((prot, lig, pock)).view(-1)
            loss = criterion(output, y)
            loss.backward()
            classic_optimizer.step()
            quantum_optimizer.step()
            
            current_loss = loss.item()
            total_loss += current_loss
            
            # Генерируем визуальную полоску лосса
            l_bar = get_loss_bar(current_loss)
            
            # Выводим в tqdm: текущий лосс, средний и нашу полоску
            avg_loss = total_loss / (i + 1)
            pbar.set_postfix_str(f"Loss: {current_loss:.4f} {l_bar} Avg: {avg_loss:.4f}")

        # Валидация метрик (RMSE, Pearson R, CI) в конце эпохи
        rmse, r_val, ci_val, preds, targets = evaluator.evaluate(val_loader)
        model.history['train_loss'].append(avg_loss)
        model.history['val_rmse'].append(rmse)
        model.history['val_pearson'].append(r_val)
        model.history['val_ci'].append(ci_val)
        if r_val >= max(model.history['val_pearson']):
            model.history['best_y_true'] = targets.tolist()
            model.history['best_y_pred'] = preds.tolist()

        with open(f"{exp_dir}/history.json", 'w') as f:
            json.dump(model.history, f, indent=4)
        torch.save(model.state_dict(), f"{exp_dir}/model_epoch_{epoch}.pt")

        # Печатаем итоги эпохи (в статье используются именно эти метрики [cite: 515, 516, 521])
        print(f"   ∟ Valid: RMSE {rmse:.4f} | R {r_val:.4f} | CI {ci_val:.4f}")
        print("-" * 60)

        if r_val > best_val_r:
            best_val_r = r_val
            best_epoch = (epoch + 1)
            torch.save(model.state_dict(), f"{exp_dir}/best_model.pt")
            print(f"--- NEW BEST R: {best_val_r:.4f} (Saved to best_model.pt) ---")

    evaluator.plot_history(exp_dir, show=show_plots, save=save_plots)

    print("\n================ ФИНАЛЬНЫЙ ТЕСТ (CORE SET) ================")
    # Подгружаем веса лучшей эпохи (в идеале нужно написать логику загрузки лучшего .pt,
    # но пока протестируем на весах последней эпохи)
    best_model_path = f"{exp_dir}/best_model.pt"
    if os.path.exists(best_model_path):
        model.load_state_dict(torch.load(best_model_path))
        print(f"Успешно загружены веса лучшей эпохи {best_epoch} из {best_model_path}")

    test_rmse, test_r, test_ci, test_preds, test_targets = evaluator.evaluate(test_loader)
    print(f"FINAL TEST -> RMSE: {test_rmse:.4f} | Pearson R: {test_r:.4f} | CI: {test_ci:.4f}")
    
    # Сохраняем результаты теста
    with open(f"{exp_dir}/test_results.json", 'w') as f:
        json.dump({
            "RMSE": test_rmse,
            "Pearson_R": test_r,
            "CI": test_ci
        }, f, indent=4)

Используем Loss: RankingMSELoss


In [10]:
train(10, show_plots=True, save_plots=True)

Starting training on cpu...
!!! ФАЙЛЫ СОХРАНЯЮТСЯ СЮДА: runs/GGG_GATv2_AdamW_SGD_20260418_162436 !!!


Epoch 1/10:   0%|          | 0/101 [00:01<?, ?batch/s]


RuntimeError: result type Float can't be cast to the desired output type Long